# L01 — When and Why to Simulate

**Module**: M01 | **Chapters**: 1–2 | **Lectures**: L01–L02

## Learning Objectives
By the end of this notebook you will be able to:
1. Classify a system along the three taxonomy axes (discrete/continuous, stochastic/deterministic, static/dynamic).
2. Identify when simulation is appropriate vs. when an analytical model suffices.
3. Explain Jensen's inequality and why averaging out randomness leads to wrong answers.
4. Use Python to compute M/M/1 analytical formulas and compare to a deterministic approximation.

---
> **Think → Trace → Code → Experiment → Interpret → Communicate**

This is the first lab of the course. There is no SimPy yet — we work with formulas and small Python scripts to build intuition before writing any simulation code.
---

## 1. The Three Classification Axes

Every simulation model lives somewhere in a three-dimensional space:

| Axis | Pole A | Pole B |
|---|---|---|
| State | **Discrete** — jumps at events | **Continuous** — evolves smoothly |
| Randomness | **Stochastic** — at least one random input | **Deterministic** — all inputs fixed |
| Time | **Dynamic** — sequence of events matters | **Static** — time plays no role |

**Exercise 1.1** — For each system below, mark each axis and name the most appropriate modeling paradigm.
Discuss with your table partner before checking the answers in Ch 2.

In [ ]:
# Classification exercise — fill in the blanks
systems = [
    "Supermarket checkout with Poisson arrivals and exponential service",
    "Chemical reactor with temperature governed by a differential equation",
    "Factory producing fixed 500 units per 8-hour shift, no variability",
    "Monte Carlo estimate of pi (throw darts at a unit square)",
    "Spread of flu through a city: each person decides to mask based on neighbors",
]

# Your answers (replace '?' with 'Discrete'/'Continuous', 'Stochastic'/'Deterministic', 'Static'/'Dynamic')
answers = [
    {'state': '?', 'randomness': '?', 'time': '?', 'paradigm': '?'},  # supermarket
    {'state': '?', 'randomness': '?', 'time': '?', 'paradigm': '?'},  # reactor
    {'state': '?', 'randomness': '?', 'time': '?', 'paradigm': '?'},  # factory
    {'state': '?', 'randomness': '?', 'time': '?', 'paradigm': '?'},  # Monte Carlo
    {'state': '?', 'randomness': '?', 'time': '?', 'paradigm': '?'},  # flu spread
]

for sys, ans in zip(systems, answers):
    print(f"System : {sys[:60]}...")
    print(f"  State={ans['state']}, Randomness={ans['randomness']}, Time={ans['time']}")
    print(f"  Paradigm: {ans['paradigm']}")
    print()

## 2. Jensen's Inequality: Why Randomness Can't Be Averaged Away

The M/M/1 queue (Poisson arrivals, exponential service, 1 server) has exact analytical formulas:
$$W_q = \frac{\lambda}{\mu(\mu - \lambda)}, \quad \rho = \frac{\lambda}{\mu}$$

A naive analyst says: "Instead of sampling from an Exponential distribution, I'll just use the mean service time 1/μ every time." This produces a **deterministic M/D/1** model.

What does the deterministic model predict?

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

mu = 10.0   # service rate (customers per hour)

rho_values = np.linspace(0.05, 0.95, 200)
lam_values = rho_values * mu

# M/M/1: Wq = lam / (mu*(mu - lam))
Wq_mm1 = lam_values / (mu * (mu - lam_values))

# M/D/1: Wq = rho^2 / (2*lam*(1-rho)) = rho / (2*(mu-lam))
Wq_md1 = rho_values / (2.0 * (mu - lam_values))

# Deterministic model: if service is exactly 1/mu, no queueing occurs while rho < 1
# A server with fixed rate mu and fixed interarrival 1/lam:
# customers never overlap -> Wq = 0 if rho < 1
Wq_det = np.zeros_like(rho_values)   # zero wait predicted

fig, ax = plt.subplots(figsize=(8, 5))
ax.plot(rho_values, Wq_mm1 * mu, 'b-',  lw=2, label='M/M/1 (exponential svc)')
ax.plot(rho_values, Wq_md1 * mu, 'g--', lw=2, label='M/D/1 (deterministic svc) = ½ M/M/1')
ax.plot(rho_values, Wq_det,      'r:',  lw=2, label='Naïve deterministic: Wq = 0')
ax.set_xlabel('Utilisation ρ = λ/μ')
ax.set_ylabel('Normalised wait μWq')
ax.set_title("Jensen's Inequality: Randomness Can't Be Averaged Away")
ax.set_xlim(0, 1)
ax.set_ylim(0, 12)
ax.legend()
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

print("At ρ = 0.8:")
rho08 = 0.8
lam08 = rho08 * mu
print(f"  M/M/1 Wq = {lam08 / (mu*(mu-lam08)):.3f} hr = {lam08/(mu*(mu-lam08))*60:.1f} min")
print(f"  M/D/1 Wq = {rho08/(2*(mu-lam08)):.3f} hr = {rho08/(2*(mu-lam08))*60:.1f} min")
print(f"  Naive     = 0 min  (catastrophically wrong)")

**Key takeaway**: The deterministic model predicts *zero* wait whenever ρ < 1. The M/M/1 model predicts substantial waits — and the M/D/1 (half-way) shows that even switching from exponential to deterministic service *halves* the wait at the same load. 

Randomness is not a nuisance to be averaged away — **variability causes congestion**.

## 3. When Is Simulation Necessary?

Use this decision tree (from Ch 1) to evaluate each scenario below.

```
Does an exact analytical formula exist?
  YES → Use it. (M/M/1, M/M/c, G/G/1 bounds, inventory EOQ...)
  NO  ↓
Is the system deterministic?
  YES → LP / MIP / scheduling optimization
  NO  ↓
Does time (sequence of events) matter?
  NO  → Monte Carlo / static simulation
  YES ↓
Does state change continuously?
  YES → ODE / system dynamics / hybrid
  NO  → Discrete-event simulation (DES)  ← this course
```

In [ ]:
# Decision exercise — rate each scenario
# Mark: 'formula', 'Monte Carlo', 'ODE/SD', 'DES', or 'ABM'

scenarios = [
    "M/M/2 call center — compute mean wait time",
    "Estimate the probability that a portfolio loses >10% in a year",
    "Hospital ED: 4 types of patients, time-varying arrivals, 6 service stages",
    "Water pressure in a pipe network (Hazen-Williams equations)",
    "Traffic: cars decide lane based on their neighbors' speed",
    "Single machine: fixed 100 parts/shift, no variability",
]

your_answers = [
    '?',   # M/M/2
    '?',   # portfolio
    '?',   # hospital ED
    '?',   # water network
    '?',   # traffic
    '?',   # fixed machine
]

for s, a in zip(scenarios, your_answers):
    print(f"{s}\n  → {a}\n")

## 4. The 12-Step Simulation Study

From Table 1.1 in the textbook. For each step below, identify what artifact is produced.
This exercise prepares you for the semester-long project.

| Step | Name | What do you produce? |
|------|------|---------------------|
| 1  | Problem formulation | Study objective (1–2 sentences) |
| 2  | Setting objectives and overall project plan | … |
| 3  | Model conceptualization | Conceptual model document (HW-01) |
| 4  | Data collection | … |
| 5  | Model translation | … |
| 6  | Verification | … |
| 7  | Validation | … |
| 8  | Experimental design | … |
| 9  | Production runs | … |
| 10 | Output analysis | … |
| 11 | More runs? | … |
| 12 | Documentation and reporting | … |

In [ ]:
# Fill in the 'What do you produce?' column
artifacts = [
    "Study objective: one sentence stating what decision the simulation will inform",
    "?",  # step 2
    "Conceptual model: entity table, resource table, event list, state variables, assumptions",
    "?",  # step 4
    "?",  # step 5
    "?",  # step 6
    "?",  # step 7
    "?",  # step 8
    "?",  # step 9
    "?",  # step 10
    "?",  # step 11
    "?",  # step 12
]
steps = [
    "Problem formulation", "Setting objectives", "Model conceptualization",
    "Data collection", "Model translation", "Verification", "Validation",
    "Experimental design", "Production runs", "Output analysis",
    "More runs?", "Documentation and reporting",
]
for i, (step, art) in enumerate(zip(steps, artifacts), 1):
    print(f"{i:2d}. {step:30s} → {art}")

---
## Try It Yourself

1. **Jensen's inequality at extremes**: What does the M/M/1 formula predict as ρ→0? As ρ→1? Compute Wq for ρ ∈ {0.01, 0.5, 0.9, 0.99, 0.999} with μ=10. Plot on a log scale. At what ρ does the wait first exceed 1 hour?

2. **Classify your own system**: Think of a real system you interact with daily (a coffee cart, a rideshare app, a dining hall). Classify it along all three axes and name the appropriate modeling paradigm. Write 3 sentences justifying your choice.

3. **M/D/1 vs M/M/1**: A hospital imaging suite has μ=4 scans/hour. Arrivals are Poisson at λ=3/hour. The radiologist claims "MRI scans always take exactly 15 minutes." Compute Wq under M/D/1 and M/M/1 assumptions. What is the percentage error of using M/M/1 when service is actually deterministic?